# Scenario 1: Retail Inventory Analysis
In this lab, you will work on three separate real-world scenarios. Each scenario will have a task for you to complete, along with a discussion component highlighting differences in generated code based on the individual prompts used. The focus is on learning how different prompting strategies can lead to various generated solutions.

## Your Prompt
**Initial Prompt Strategy (Context Setting & Framework):**
> "Act as a senior data analyst. I have a pandas DataFrame with retail inventory data. The columns are: `product_id`, `category`, `stock_level`, `last_restock_date` (datetime), `sales_last_30_days` (int), `supplier_lead_time` (days), and `unit_cost` (float). 
> 
> Please write Python code to:
> 1. Calculate inventory turnover rates (use sales_last_30_days / stock_level as a proxy).
> 2. Identify slow-moving items (items where sales are low but stock is high).
> 3. Predict potential stockouts (items that will run out of stock before the supplier lead time based on the current sales rate).
> 4. Create a few visualizations to present these insights to the store manager.
> 
> Please include basic error handling for missing data and divide-by-zero scenarios."

## Generated Code

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_inventory(df):
    # Error Handling: Check if dataframe is empty and handle missing values
    if df.empty:
        raise ValueError("The input DataFrame is empty.")
    
    df = df.copy()
    df.fillna({'stock_level': 0, 'sales_last_30_days': 0}, inplace=True)
    
    # 1. Calculate inventory turnover rates (handling division by zero)
    df['turnover_rate'] = np.where(df['stock_level'] > 0, 
                                   df['sales_last_30_days'] / df['stock_level'], 
                                   0)
    
    # 2. Identify slow-moving items (e.g., Turnover < 0.1 and Stock > 50)
    df['is_slow_moving'] = (df['turnover_rate'] < 0.1) & (df['stock_level'] > 50)
    slow_moving = df[df['is_slow_moving']]
    
    # 3. Predict potential stockouts
    # Calculate daily sales rate
    df['daily_sales_rate'] = df['sales_last_30_days'] / 30
    
    # Days until stock runs out
    df['days_until_stockout'] = np.where(df['daily_sales_rate'] > 0,
                                         df['stock_level'] / df['daily_sales_rate'],
                                         np.inf)
    
    # Stockout risk if days until stockout is less than supplier lead time
    df['stockout_risk'] = df['days_until_stockout'] <= df['supplier_lead_time']
    stockout_items = df[df['stockout_risk']]
    
    # 4. Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Turnover rate by category
    sns.boxplot(x='category', y='turnover_rate', data=df, ax=axes[0])
    axes[0].set_title('Inventory Turnover Rate by Category')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Plot 2: Count of Stockout Risk Items by Category
    sns.countplot(x='category', data=stockout_items, ax=axes[1], palette='Reds_r')
    axes[1].set_title('Items at Risk of Stockout by Category')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return slow_moving, stockout_items

# Example Usage:
# slow_items, risky_items = analyze_inventory(my_dataframe)

## Follow-up Prompts
**Refinement & Testing:**
While the initial code was good, I noticed that `last_restock_date` wasn't being utilized, and the visual for slow-moving items was missing. I used a follow-up prompt to optimize the solution:

> *"This is a great start. However, please update the code to do the following:*
> *1. Use the `last_restock_date` to calculate "days_since_last_restock". If an item is slow-moving AND hasn't been restocked in over 90 days, flag it as 'Dead Stock'.*
> *2. Change the visualizations: I want a scatter plot showing `stock_level` vs `sales_last_30_days` with 'Dead Stock' highlighted in red.*
> *3. Ensure proper datetime conversion error handling for the date column."*

## Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_inventory_optimized(df):
    if df.empty:
        raise ValueError("The input DataFrame is empty.")
    
    df = df.copy()
    
    # Error Handling: Robust Date Conversion and Missing Values
    try:
        df['last_restock_date'] = pd.to_datetime(df['last_restock_date'], errors='coerce')
    except Exception as e:
        print(f"Warning: Date conversion failed: {e}")
        
    df.fillna({'stock_level': 0, 'sales_last_30_days': 0, 'supplier_lead_time': 0}, inplace=True)
    
    # 1. Turnover Rate
    df['turnover_rate'] = np.where(df['stock_level'] > 0, 
                                   df['sales_last_30_days'] / df['stock_level'], 0)
    
    # 2. Daily Sales & Stockout Risk
    df['daily_sales_rate'] = df['sales_last_30_days'] / 30
    df['days_until_stockout'] = np.where(df['daily_sales_rate'] > 0,
                                         df['stock_level'] / df['daily_sales_rate'], np.inf)
    df['stockout_risk'] = df['days_until_stockout'] <= df['supplier_lead_time']
    
    # 3. Slow-moving & Dead Stock Calculation using Dates
    current_date = pd.Timestamp.now()
    df['days_since_last_restock'] = (current_date - df['last_restock_date']).dt.days
    
    # Condition: High stock, low sales
    is_slow = (df['turnover_rate'] < 0.1) & (df['stock_level'] > 50)
    # Condition: Slow moving AND hasn't been restocked recently
    df['inventory_status'] = np.where(is_slow & (df['days_since_last_restock'] > 90), 
                                      'Dead Stock', 
                                      np.where(is_slow, 'Slow Moving', 'Healthy'))
    
    # 4. Visualizations
    plt.figure(figsize=(10, 6))
    
    # Scatter plot highlighting Dead Stock
    sns.scatterplot(data=df, x='stock_level', y='sales_last_30_days', 
                    hue='inventory_status', 
                    palette={'Dead Stock': 'red', 'Slow Moving': 'orange', 'Healthy': 'green'},
                    alpha=0.7)
    
    plt.title('Inventory Health: Stock Level vs. 30-Day Sales')
    plt.xlabel('Current Stock Level')
    plt.ylabel('Sales (Last 30 Days)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()
    
    return df

# Example Usage:
# final_df = analyze_inventory_optimized(my_dataframe)

## Discussion Questions to Answer

**1. How did different prompts handle date calculations?**
* **Initial Prompt:** The initial broad prompt completely ignored the `last_restock_date` column because I didn't explicitly instruct the AI on *how* to use it to solve the business problem. 
* **Refined Prompt:** Once I explicitly asked to calculate "days since last restock" to find dead stock, the AI handled the date calculations effectively using `pd.to_datetime` and subtracting it from `pd.Timestamp.now()`. It highlighted the importance of explicitly defining the business logic you want applied to specific data types in your prompts.

**2. What visualization approaches were suggested?**
* The broad prompt defaulted to standard exploratory data analysis (EDA) charts: a boxplot to show the distribution of turnover rates across categories, and a basic bar chart (countplot) to show the volume of stockout risks.
* By using a refined prompt, I was able to guide the AI to generate a much more actionable visualization: a colored scatter plot highlighting "Dead Stock" vs. "Healthy" items, which is far more useful for a store manager trying to identify specific problem items at a glance.

**3. How was error handling implemented?**
* In the initial prompt, I asked for "basic error handling." The AI responded by checking for an empty dataframe (`df.empty`) and using `.fillna()` to handle NaN values in numeric columns. It also smartly used `np.where()` to prevent `ZeroDivisionError` when dividing by `stock_level`.
* In the refined prompt, because I was now working with dates, the AI added a `try-except` block around the date conversion step and used `errors='coerce'` inside `pd.to_datetime`, ensuring that improperly formatted dates would safely turn into `NaT` (Not a Time) rather than crashing the entire script.